<a href="https://colab.research.google.com/github/Loubna1101/Qwen3-TTS-Darija-LoRa/blob/main/notebooks/inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q qwen-tts peft soundfile gradio huggingface_hub

In [ ]:
from huggingface_hub import snapshot_download

model_repo = "loubna1101/Qwen3-TTS-Darija-LoRa"

ckpt_dir = snapshot_download(
    repo_id=model_repo,
    repo_type="model"
)

print("Downloaded to:", ckpt_dir)

In [ ]:
import json
import torch
from peft import PeftModel
from qwen_tts.inference.qwen3_tts_model import Qwen3TTSModel

BASE_MODEL = "Qwen/Qwen3-TTS-12Hz-1.7B-Base"

qwen3tts = Qwen3TTSModel.from_pretrained(
    BASE_MODEL,
    attn_implementation="eager",
)

qwen3tts.model.talker.model = PeftModel.from_pretrained(
    qwen3tts.model.talker.model,
    f"{ckpt_dir}/talker_lora"
)

if hasattr(qwen3tts.model.talker.model, "merge_and_unload"):
    qwen3tts.model.talker.model = qwen3tts.model.talker.model.merge_and_unload()

spk = torch.load(f"{ckpt_dir}/speaker_embedding.pt", map_location="cpu")
speaker_embedding = spk["embedding"]
speaker_id = spk["speaker_id"]

with torch.no_grad():
    emb = qwen3tts.model.talker.model.codec_embedding.weight
    emb[speaker_id] = speaker_embedding.to(emb.device).to(emb.dtype)

with open(f"{ckpt_dir}/config.json", "r", encoding="utf-8") as f:
    saved_config = json.load(f)

qwen3tts.model.tts_model_type = saved_config["tts_model_type"]
qwen3tts.model.config.tts_model_type = saved_config["tts_model_type"]

saved_talker_config = saved_config.get("talker_config", {})

if hasattr(qwen3tts.model.config, "talker_config"):
    if "spk_id" in saved_talker_config:
        qwen3tts.model.config.talker_config.spk_id = saved_talker_config["spk_id"]
    if "spk_is_dialect" in saved_talker_config:
        qwen3tts.model.config.talker_config.spk_is_dialect = saved_talker_config["spk_is_dialect"]

if hasattr(qwen3tts.model, "talker_config") and hasattr(qwen3tts.model.talker_config, "__dict__"):
    if "spk_id" in saved_talker_config:
        qwen3tts.model.talker_config.spk_id = saved_talker_config["spk_id"]
    if "spk_is_dialect" in saved_talker_config:
        qwen3tts.model.talker_config.spk_is_dialect = saved_talker_config["spk_is_dialect"]

speaker_names = set(saved_talker_config.get("spk_id", {}).keys())
qwen3tts._supported_speakers_set = lambda: speaker_names

print("Supported speakers:", qwen3tts._supported_speakers_set())
print("tts_model_type:", qwen3tts.model.tts_model_type)

In [ ]:
audio = qwen3tts.generate_custom_voice(
    text=".بارح خرجت نتمشى شوية فالحومة باش نبدل الجو",
    speaker="darija_speaker"
)

In [ ]:
import soundfile as sf
from IPython.display import Audio

waveform = audio[0][0].squeeze().astype("float32")
sr = audio[1]

sf.write("/content/audio.wav", waveform, sr)

Audio("/content/audio.wav")